Confiming the copy landed correctly — 3.44 GB, sitting in raw, matches the expected size for this file. So the data transfer is fully verified and done; no need to double-check that further.

In [1]:
from google.colab import drive
drive.mount('/content/drive')



Mounted at /content/drive


Below we are checking: every key expected from the data dictionary structure showed up: all six categories (A, T, W, X_s, X_v, Y) split into _dev/_test, with _var name-list counterparts for everything except Y (which makes sense — Y is just the RUL target column, a single number per row, so there's no set of variable names to look up). No surprises or missing pieces here, which is a good sign the file is exactly what the proposal was built around.

In [2]:
import h5py
path = '/content/drive/MyDrive/edge-ai-fault-diagnosis-aerospace/data/raw/N-CMAPSS_DS03-012.h5'
f = h5py.File(path, 'r')
print(list(f.keys()))

['A_dev', 'A_test', 'A_var', 'T_dev', 'T_test', 'T_var', 'W_dev', 'W_test', 'W_var', 'X_s_dev', 'X_s_test', 'X_s_var', 'X_v_dev', 'X_v_test', 'X_v_var', 'Y_dev', 'Y_test']


Next step: check the actual shapes and column counts against what docs/data-dictionary.md currently states.  

In [3]:
#Checking the actual shapes and columns counts
for key in f.keys():
    print(key, f[key].shape, f[key].dtype)

A_dev (5571277, 4) float64
A_test (4251560, 4) float64
A_var (4,) |S5
T_dev (5571277, 10) float64
T_test (4251560, 10) float64
T_var (10,) |S12
W_dev (5571277, 4) float64
W_test (4251560, 4) float64
W_var (4,) |S4
X_s_dev (5571277, 14) float64
X_s_test (4251560, 14) float64
X_s_var (14,) |S4
X_v_dev (5571277, 14) float64
X_v_test (4251560, 14) float64
X_v_var (14,) |S5
Y_dev (5571277, 1) int64
Y_test (4251560, 1) int64


Above

Above output - Everything matches. Every column count lines up exactly with what docs/data-dictionary.md assumes: W at 4 columns, X_s and X_v at 14 each, T at 10, A at 4, and Y at 1 (the RUL target, stored as int64 rather than float, which makes sense since it's a whole-number remaining-cycles count). No corrections needed to the data dictionary — it was written accurately.

The row counts are worth noting too: dev has 5,571,277 rows and test has 4,251,560, which adds up to 9,822,837 total — right in line with the proposal's "~9.8M records" figure. That's a second confirmation the file matches what the project was designed around.

Next: Decode the variable names, so you know exactly what each column represents rather than just a column index:

In [4]:
#ecode the variable names, so you know exactly what each column represents rather than just a column index:
w_names = [n.decode() if isinstance(n, bytes) else n for n in f['W_var'][:]]
xs_names = [n.decode() if isinstance(n, bytes) else n for n in f['X_s_var'][:]]
print("W columns:", w_names)
print("X_s columns:", xs_names)

W columns: ['alt', 'Mach', 'TRA', 'T2']
X_s columns: ['T24', 'T30', 'T48', 'T50', 'P15', 'P2', 'P21', 'P24', 'Ps30', 'P40', 'P50', 'Nf', 'Nc', 'Wf']


Above output:That's the full column mapping — 4 flight-condition variables (alt, Mach, TRA, T2) and 14 real sensor names (T24 through Wf — a mix of temperatures, pressures, and rotational speeds, matching standard turbofan sensor naming conventions). This is genuinely useful to have confirmed

Next: heck — unit and cycle counts. Run:

In [5]:
import numpy as np

a_dev = f['A_dev'][:]
units = np.unique(a_dev[:, 0])
print("Number of units:", len(units))
print("Unit IDs:", units)

Number of units: 9
Unit IDs: [1. 2. 3. 4. 5. 6. 7. 8. 9.]


Above results implication for my praxis proposal: This is an important finding — DS03 has 9 units, not the ~100 the proposal describes. That's not an error in your download or a mistake in your commands; it reflects how N-CMAPSS is actually structured. NASA split the full N-CMAPSS collection across eight separate files (DS01 through DS08), each covering a different subset of engine units, flight classes, and fault-mode combinations rather than every file containing the whole dataset. The "100 units, 7 failure modes" figure in your proposal almost certainly describes the entire N-CMAPSS collection across all eight DS0x files combined, not what's inside DS03 alone.

This is worth resolving properly rather than guessing, since it affects how you describe your dataset in the actual work. A couple of ways to pin it down:

The NASA example notebook you already have in your notebooks/ folder (N_CMAPSS_Example_data_loading_and_exploration.ipynb) likely documents exactly which units and fault modes are in DS03 specifically — worth opening it and checking, since it's the authoritative reference for this dataset.

Separately, it's worth checking the exact wording in your original proposal/pptx around "100 units, 7 failure modes" — if it was describing N-CMAPSS in general rather than committing specifically to DS03, then working with 9 units in DS03 is completely fine and expected; the docs just need a small correction to say "9 units from the DS03 subset" rather than implying all 100 units are available in this file.

Next: One more useful number before closing this out — how many flight cycles does each of these 9 units actually have? Run:

In [6]:
example_unit = units[0]
mask = a_dev[:, 0] == example_unit
print(f"Unit {example_unit} has {mask.sum()} rows (flight-cycle samples)")

Unit 1.0 has 308489 rows (flight-cycle samples)


Above result:That confirms it — 308,489 rows for unit 1 alone, and this is worth interpreting correctly: this is not 308,489 flight cycles. N-CMAPSS records sensor readings at a high sampling rate across each flight, so a single flight cycle spans many rows of time-series data. To get the actual cycle count, we need to know which column in A_dev holds the cycle number — right now we've been assuming column 0 is unit ID, but we haven't actually confirmed the column order.

Next steps:Let's decode A_var the same way you decoded W_var and X_s_var, so we know exactly what each of the 4 columns in A_dev means:

In [7]:
a_names = [n.decode() if isinstance(n, bytes) else n for n in f['A_var'][:]]
print("A columns:", a_names)

A columns: ['unit', 'cycle', 'Fc', 'hs']


Above result: Exactly as expected — unit, cycle, Fc (flight class), hs (health state). Now we can get the real cycle count using the correct column:

In [8]:
cycle_col = a_names.index('cycle')
cycles_unit1 = np.unique(a_dev[mask, cycle_col])
print(f"Unit {example_unit} has {len(cycles_unit1)} distinct flight cycles")
print("Flight classes present:", np.unique(a_dev[mask, a_names.index('Fc')]))
print("Health states present:", np.unique(a_dev[mask, a_names.index('hs')]))

Unit 1.0 has 72 distinct flight cycles
Flight classes present: [1.]
Health states present: [0. 1.]


Above Resultts: That's a clean, sensible result. Unit 1 flew 72 cycles, all within flight class 1 (N-CMAPSS groups flights into three length classes — 1 is typically the shortest), and its health state moved between 1 (healthy) and 0 (degraded) across its life — exactly the pattern you'd want for a fault-diagnosis dataset, since it means this unit's data actually spans both normal and degraded operation rather than being all-healthy or all-faulty.

On the "7 failure modes" figure from the proposal: hs here is just a binary healthy/unhealthy flag, not a specific fault-type label, so that number almost certainly refers to something at a higher level — most likely the different fault-mode combinations NASA built into the eight separate DS0x files (DS03 representing one particular combination), rather than 7 distinct labels living inside this one file. That's worth confirming against the NASA example notebook or the Chao et al. citation rather than guessing further from the data alone — not something to resolve right now, just flagging it as a small follow-up before you write it into your final docs.

Let's run the verification pass. Add these as new cells in your 01_eda.ipynb notebook (right after your existing DS03 checks) — same pattern, just pointed at the two new files.
Check DS01:

In [9]:
path_ds01 = '/content/drive/MyDrive/edge-ai-fault-diagnosis-aerospace/data/raw/N-CMAPSS_DS01-005.h5'
f1 = h5py.File(path_ds01, 'r')
print("Keys:", list(f1.keys()))
for key in f1.keys():
    print(key, f1[key].shape, f1[key].dtype)

Keys: ['A_dev', 'A_test', 'A_var', 'T_dev', 'T_test', 'T_var', 'W_dev', 'W_test', 'W_var', 'X_s_dev', 'X_s_test', 'X_s_var', 'X_v_dev', 'X_v_test', 'X_v_var', 'Y_dev', 'Y_test']
A_dev (4906636, 4) float64
A_test (2735232, 4) float64
A_var (4,) |S5
T_dev (4906636, 10) float64
T_test (2735232, 10) float64
T_var (10,) |S12
W_dev (4906636, 4) float64
W_test (2735232, 4) float64
W_var (4,) |S4
X_s_dev (4906636, 14) float64
X_s_test (2735232, 14) float64
X_s_var (14,) |S4
X_v_dev (4906636, 14) float64
X_v_test (2735232, 14) float64
X_v_var (14,) |S5
Y_dev (4906636, 1) int64
Y_test (2735232, 1) int64


Results above: Clean match — every column count for DS01 lines up exactly with DS03: W=4, X_s=14, X_v=14, T=10, A=4, Y=1. Same key names too. That's exactly what preprocessing.py needs — all three files can be loaded and combined with identical logic since their structure is consistent.

Row counts: dev has 4,906,636 rows, test has 2,735,232 — a total of 7,641,868 for this file alone.

#Now run the DS08a check:


In [10]:
path_ds08a = '/content/drive/MyDrive/edge-ai-fault-diagnosis-aerospace/data/raw/N-CMAPSS_DS08a-009.h5'
f8 = h5py.File(path_ds08a, 'r')
print("Keys:", list(f8.keys()))
for key in f8.keys():
    print(key, f8[key].shape, f8[key].dtype)

Keys: ['A_dev', 'A_test', 'A_var', 'T_dev', 'T_test', 'T_var', 'W_dev', 'W_test', 'W_var', 'X_s_dev', 'X_s_test', 'X_s_var', 'X_v_dev', 'X_v_test', 'X_v_var', 'Y_dev', 'Y_test']
A_dev (4885389, 4) float64
A_test (3722997, 4) float64
A_var (4,) |S5
T_dev (4885389, 10) float64
T_test (3722997, 10) float64
T_var (10,) |S12
W_dev (4885389, 4) float64
W_test (3722997, 4) float64
W_var (4,) |S4
X_s_dev (4885389, 14) float64
X_s_test (3722997, 14) float64
X_s_var (14,) |S4
X_v_dev (4885389, 14) float64
X_v_test (3722997, 14) float64
X_v_var (14,) |S5
Y_dev (4885389, 1) int64
Y_test (3722997, 1) int64


Results above: Another clean match — DS08a's columns are identical in structure to DS01 and DS03: W=4, X_s=14, X_v=14, T=10, A=4, Y=1, same key names throughout. All three files are now confirmed structurally consistent, which means preprocessing.py can load and combine them with one shared loading function rather than needing special-case logic per file.

Row counts across all three: DS01 (7,641,868) + DS03 (9,822,837) + DS08a (8,608,386) = 26,073,091 total rows once combined. That's a substantial training set for the MA1DCNN.

In [11]:
a1 = f1['A_dev'][:]
print("DS01 dev units:", len(np.unique(a1[:, 0])))

a8 = f8['A_dev'][:]
print("DS08a dev units:", len(np.unique(a8[:, 0])))

DS01 dev units: 6
DS08a dev units: 9
